# Improved Kaggle Training Notebook (Anti-Overfitting)

This notebook trains the retinal model with improvements:
- Correct ResNet50V2 input scaling
- Smaller `LNN` head (`256,128`)
- `val_auc`-based early stopping/checkpointing
- Lower fine-tuning aggressiveness (`unfreeze_layers=4`)
- Controlled regularization (`label_smoothing=0.08`, AdamW weight decay)

In [ ]:
# Kaggle setup for dataset-only runs (no repo required)
import os, glob
from pathlib import Path

# Try likely mounted ODIR roots
initial_candidates = [
    '/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/ODIR-5K',
    '/kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K',
 ]
initial_candidates += glob.glob('/kaggle/input/**/ODIR-5K', recursive=True)

def score_root(root):
    if not os.path.isdir(root):
        return -1
    csvs = glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
    xls = glob.glob(os.path.join(root, '**', '*.xlsx'), recursive=True)
    has_pre = bool(glob.glob(os.path.join(root, '**', 'preprocessed_images'), recursive=True))
    has_train = bool(glob.glob(os.path.join(root, '**', 'Training Images'), recursive=True))
    return (3 if csvs else 0) + (2 if xls else 0) + (2 if has_pre else 0) + (1 if has_train else 0)

# Include parent folders too (handles nested ODIR-5K/ODIR-5K layouts)
expanded = set()
for c in initial_candidates:
    if os.path.isdir(c):
        expanded.add(c)
        expanded.add(str(Path(c).parent))
        expanded.add(str(Path(c).parent.parent))

best = sorted([(score_root(r), r) for r in expanded], reverse=True)
DATASET_ROOT = best[0][1] if best and best[0][0] > 0 else None

if DATASET_ROOT is None:
    raise FileNotFoundError('ODIR dataset folder not found under /kaggle/input. Please add the dataset in Kaggle Inputs.')

%cd /kaggle/working
print('Using DATASET_ROOT:', DATASET_ROOT)
print('Top-level files/folders:')
for p in sorted(glob.glob(os.path.join(DATASET_ROOT, '*')))[:30]:
    print('-', os.path.basename(p))

In [ ]:
# Imports + self-contained fallback implementation (no src/ dependency)
import os, json, math, glob, ast
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, hamming_loss,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
 )

DISEASE_LABELS = ['N','D','G','C','A','H','M','O']

def _find_image_file(img_name, img_root):
    p = os.path.join(img_root, img_name)
    if os.path.isfile(p):
        return p
    # fallback recursive search by basename
    hits = glob.glob(os.path.join(img_root, '**', os.path.basename(img_name)), recursive=True)
    return hits[0] if hits else None

def load_odir_dataset(csv_path, img_dir, img_size=224, sample_fraction=1.0):
    df = pd.read_csv(csv_path)
    if sample_fraction < 1.0:
        df = df.sample(frac=sample_fraction, random_state=42)

    images, targets = [], []
    has_multilabel_cols = all(c in df.columns for c in DISEASE_LABELS)

    for _, row in df.iterrows():
        # Case 1: canonical format with filename + serialized target list
        if 'filename' in df.columns and 'target' in df.columns:
            fname = row['filename']
            target = ast.literal_eval(row['target']) if isinstance(row['target'], str) else row['target']
            img_path = _find_image_file(str(fname), img_dir)
            if img_path is None:
                continue
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size)).astype(np.float32) / 255.0
            images.append(img)
            targets.append(np.array(target, dtype=np.float32))
            continue

        # Case 2: AndrewMVD-style format with Left-Fundus/Right-Fundus + N..O columns
        if has_multilabel_cols and ('Left-Fundus' in df.columns or 'Right-Fundus' in df.columns):
            y = row[DISEASE_LABELS].values.astype(np.float32)
            for col in ['Left-Fundus', 'Right-Fundus']:
                if col in df.columns and isinstance(row[col], str):
                    img_path = _find_image_file(row[col], img_dir)
                    if img_path is None:
                        continue
                    img = cv2.imread(img_path)
                    if img is None:
                        continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (img_size, img_size)).astype(np.float32) / 255.0
                    images.append(img)
                    targets.append(y)

    X = np.array(images, dtype=np.float32)
    y = np.array(targets, dtype=np.float32)
    if len(X) == 0:
        raise ValueError('No images loaded. Check CSV/image folder mapping.')

    strat = None
    if len(np.unique(np.argmax(y, axis=1))) > 1:
        strat = np.argmax(y, axis=1)
    try:
        return train_test_split(X, y, test_size=0.2, random_state=42, stratify=strat)
    except ValueError:
        return train_test_split(X, y, test_size=0.2, random_state=42)

def augment_image(img, label, training=True):
    if not training:
        return img, label
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_brightness(img, max_delta=0.18)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.clip_by_value(img + tf.random.normal(tf.shape(img), stddev=0.015), 0.0, 1.0)
    return img, label

def build_tf_dataset(X, y, batch_size=32, shuffle=True, augment=True, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(1024, len(X)), seed=42)
    if repeat:
        ds = ds.repeat()
    ds = ds.map(lambda x,l: augment_image(x,l,training=augment), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def build_tf_dataset_val(X, y, batch_size=32):
    return tf.data.Dataset.from_tensor_slices((X, y)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

def compute_pos_weights(y_train, max_ratio=20.0):
    y = np.asarray(y_train, dtype=np.float64)
    pos = np.maximum(y.sum(axis=0), 1.0)
    n = float(y.shape[0])
    neg = n - y.sum(axis=0)
    return np.clip(neg / pos, 1.0, max_ratio).astype(np.float32)

def make_weighted_focal_loss(pos_weights, gamma=2.0, label_smoothing=0.08):
    pos_w = tf.constant(pos_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        if label_smoothing > 0:
            y_true = y_true * (1.0 - label_smoothing) + 0.5 * label_smoothing
        p = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -y_true * tf.math.log(p) - (1 - y_true) * tf.math.log(1 - p)
        p_t = y_true * p + (1 - y_true) * (1 - p)
        focal = tf.pow(1.0 - p_t, gamma) * bce
        w = y_true * (pos_w - 1.0) + 1.0
        return tf.reduce_mean(focal * w)
    return loss

class LiquidCell(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = units
        self.layer_norm = tf.keras.layers.LayerNormalization(name='lnn_lnorm')

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), initializer='glorot_uniform', regularizer=tf.keras.regularizers.l2(1e-4), name='w_in')
        self.w_rec = self.add_weight(shape=(self.units, self.units), initializer='glorot_uniform', regularizer=tf.keras.regularizers.l2(1e-4), name='w_rec')
        self.b = self.add_weight(shape=(self.units,), initializer='zeros', name='b')
        self.tau = self.add_weight(shape=(self.units,), initializer='ones', name='tau', constraint=tf.keras.constraints.NonNeg())
        self.layer_norm.build((None, self.units))
        self.built = True

    def call(self, inputs, states):
        prev_state = states[0]
        base_input = tf.matmul(inputs, self.w_in) + tf.matmul(prev_state, self.w_rec) + self.b
        liquid_tau = self.tau * tf.math.sigmoid(base_input)
        new_state = prev_state + (base_input - prev_state) * (liquid_tau + 1e-7)
        new_state = self.layer_norm(new_state)
        return new_state, [new_state]

def _resolve_resnet_weights(weights_mode='auto'):
    """Resolve ResNet weights for online/offline Kaggle runs."""
    cache_file = os.path.expanduser('~/.keras/models/resnet50v2_weights_tf_dim_ordering_tf_kernels_notop.h5')
    if os.path.isfile(cache_file):
        return cache_file
    if weights_mode == 'none':
        return None
    if weights_mode == 'imagenet':
        return 'imagenet'
    # auto: attempt imagenet and fallback in builder
    return 'imagenet'

def build_concept_aware_lnn(input_shape=(224,224,3), num_classes=8, lnn_units=(256,128), use_model_augmentation=False, weights_mode='auto'):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs
    if use_model_augmentation:
        x = tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal_and_vertical'),
            tf.keras.layers.RandomRotation(0.3),
            tf.keras.layers.RandomZoom(0.3),
            tf.keras.layers.RandomContrast(0.3),
            tf.keras.layers.RandomBrightness(0.2),
        ])(x)
    # ResNet50V2 scaling
    x = tf.keras.layers.Lambda(lambda t: (t * 2.0) - 1.0, name='resnetv2_input_scaling')(x)

    resolved_weights = _resolve_resnet_weights(weights_mode=weights_mode)
    try:
        base_cnn = tf.keras.applications.ResNet50V2(weights=resolved_weights, include_top=False, input_tensor=x)
        print('ResNet weights source:', resolved_weights)
    except Exception as e:
        print('Could not load ImageNet weights (offline). Falling back to random init. Error:', e)
        base_cnn = tf.keras.applications.ResNet50V2(weights=None, include_top=False, input_tensor=x)
        print('ResNet weights source: None (random init)')

    base_cnn.trainable = False
    last_conv = base_cnn.get_layer('post_relu')
    features = tf.keras.layers.GlobalAveragePooling2D(name='cnn_gap')(last_conv.output)
    features = tf.keras.layers.BatchNormalization(name='cnn_bn')(features)
    features = tf.keras.layers.Dropout(0.4, name='cnn_dropout')(features)

    h = tf.keras.layers.BatchNormalization(name='lnn_bn_in')(features)
    h = tf.keras.layers.Dropout(0.45, name='lnn_drop_in')(h)
    for i, units in enumerate(lnn_units):
        h_seq = tf.keras.layers.Lambda(lambda t: tf.expand_dims(t, axis=1), name=f'to_seq_{i}')(h)
        h = tf.keras.layers.RNN(LiquidCell(units), name=f'liquid_layer_{i}')(h_seq)
        h = tf.keras.layers.BatchNormalization(name=f'lnn_bn_{i}')(h)
        h = tf.keras.layers.Dropout(0.5, name=f'lnn_drop_{i}')(h)

    outputs = tf.keras.layers.Dense(num_classes, activation='sigmoid', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name='disease_predictions')(h)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='CNN_LNN_Concept_Aware')
    return model, last_conv.name, base_cnn

def _json_safe(obj):
    if isinstance(obj, dict): return {k:_json_safe(v) for k,v in obj.items()}
    if isinstance(obj, list): return [_json_safe(v) for v in obj]
    if isinstance(obj, (float, np.floating)):
        x = float(obj)
        return None if (math.isnan(x) or math.isinf(x)) else x
    if isinstance(obj, (np.integer,)): return int(obj)
    return obj

def binarize_predictions(y_pred, threshold=0.5):
    y_pred = np.array(y_pred, dtype=np.float64)
    thr = np.asarray(threshold, dtype=np.float64)
    if thr.ndim == 0: return (y_pred >= float(thr)).astype(np.float32)
    return (y_pred >= thr.reshape(1, -1)).astype(np.float32)

def tune_per_class_thresholds_f1(y_true, y_pred_proba, n_steps=37):
    y_true = np.asarray(y_true); y_pred_proba = np.asarray(y_pred_proba)
    n_classes = y_true.shape[1]
    y_bin = (y_true >= 0.5).astype(np.int32)
    th = np.zeros(n_classes, dtype=np.float64)
    for j in range(n_classes):
        best_t, best_f1 = 0.5, -1.0
        for t in np.linspace(0.05,0.95,n_steps):
            pred_c = (y_pred_proba[:,j] >= t).astype(np.int32)
            f1 = f1_score(y_bin[:,j], pred_c, zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        th[j] = best_t
    return th

def predict_with_tta(model, X, batch_size=16, verbose=0):
    X = np.asarray(X, dtype=np.float32)
    preds = [model.predict(X, batch_size=batch_size, verbose=verbose)]
    preds.append(model.predict(np.flip(X, axis=2), batch_size=batch_size, verbose=verbose))
    preds.append(model.predict(np.flip(X, axis=1), batch_size=batch_size, verbose=verbose))
    return np.mean(np.stack(preds, axis=0), axis=0)

def compute_all_metrics(y_true, y_pred_proba, threshold=0.5):
    y_true = np.array(y_true)
    y_true_bin = (y_true >= 0.5).astype(np.int32)
    y_pred = binarize_predictions(y_pred_proba, threshold)
    m = {}
    m['accuracy'] = float(accuracy_score(y_true_bin, y_pred))
    m['precision_macro'] = float(precision_score(y_true_bin, y_pred, average='macro', zero_division=0))
    m['precision_micro'] = float(precision_score(y_true_bin, y_pred, average='micro', zero_division=0))
    m['recall_macro'] = float(recall_score(y_true_bin, y_pred, average='macro', zero_division=0))
    m['recall_micro'] = float(recall_score(y_true_bin, y_pred, average='micro', zero_division=0))
    m['f1_macro'] = float(f1_score(y_true_bin, y_pred, average='macro', zero_division=0))
    m['f1_micro'] = float(f1_score(y_true_bin, y_pred, average='micro', zero_division=0))
    m['hamming_loss'] = float(hamming_loss(y_true_bin, y_pred))
    m['subset_accuracy'] = float(np.mean(np.all(y_true_bin == y_pred, axis=1)))
    try: m['roc_auc_macro'] = float(roc_auc_score(y_true_bin, y_pred_proba, average='macro', multi_class='ovr'))
    except Exception: m['roc_auc_macro'] = 0.0
    try: m['roc_auc_micro'] = float(roc_auc_score(y_true_bin, y_pred_proba, average='micro', multi_class='ovr'))
    except Exception: m['roc_auc_micro'] = 0.0
    try: m['average_precision_macro'] = float(average_precision_score(y_true_bin, y_pred_proba, average='macro'))
    except Exception: m['average_precision_macro'] = 0.0
    try: m['average_precision_micro'] = float(average_precision_score(y_true_bin, y_pred_proba, average='micro'))
    except Exception: m['average_precision_micro'] = 0.0
    m['precision_per_class'] = precision_score(y_true_bin, y_pred, average=None, zero_division=0).tolist()
    m['recall_per_class'] = recall_score(y_true_bin, y_pred, average=None, zero_division=0).tolist()
    m['f1_per_class'] = f1_score(y_true_bin, y_pred, average=None, zero_division=0).tolist()
    return m

def plot_training_history(history, save_dir, prefix='training'):
    os.makedirs(save_dir, exist_ok=True)
    h = history.history if hasattr(history, 'history') else {}
    fig, ax = plt.subplots(figsize=(6,4))
    if 'loss' in h: ax.plot(h['loss'], label='Train Loss')
    if 'val_loss' in h: ax.plot(h['val_loss'], label='Val Loss')
    ax.set_title('Training and Validation Loss'); ax.legend(); ax.set_xlabel('Epoch')
    plt.tight_layout(); plt.savefig(os.path.join(save_dir, f'{prefix}_loss.png'), dpi=150); plt.close()

    if 'accuracy' in h or 'val_accuracy' in h:
        fig, ax = plt.subplots(figsize=(6,4))
        if 'accuracy' in h: ax.plot(h['accuracy'], label='Train Accuracy')
        if 'val_accuracy' in h: ax.plot(h['val_accuracy'], label='Val Accuracy')
        ax.set_title('Accuracy'); ax.legend(); ax.set_xlabel('Epoch')
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f'{prefix}_accuracy.png'), dpi=150); plt.close()

    if 'auc' in h or 'val_auc' in h:
        fig, ax = plt.subplots(figsize=(6,4))
        if 'auc' in h: ax.plot(h['auc'], label='Train AUC')
        if 'val_auc' in h: ax.plot(h['val_auc'], label='Val AUC')
        ax.set_title('AUC'); ax.legend(); ax.set_xlabel('Epoch')
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f'{prefix}_auc.png'), dpi=150); plt.close()

def plot_confusion_matrices(y_true, y_pred, save_dir, labels=DISEASE_LABELS):
    os.makedirs(save_dir, exist_ok=True)
    n_classes = y_true.shape[1]
    for i,name in enumerate(labels[:n_classes]):
        cm = confusion_matrix(y_true[:,i], y_pred[:,i], labels=[0,1])
        fig, ax = plt.subplots(figsize=(4,3))
        ax.imshow(cm, cmap='Blues')
        ax.set_title(f'Confusion Matrix — {name}')
        ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['Neg','Pos']); ax.set_yticklabels(['Neg','Pos'])
        for r in range(2):
            for c in range(2):
                ax.text(c, r, str(int(cm[r,c])), ha='center', va='center')
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f'confusion_matrix_{name}.png'), dpi=150); plt.close()

def plot_aggregate_confusion(y_true, y_pred, save_dir, labels=DISEASE_LABELS):
    true_idx = np.argmax(y_true, axis=1); pred_idx = np.argmax(y_pred, axis=1)
    cm = confusion_matrix(true_idx, pred_idx, labels=range(y_true.shape[1]))
    fig, ax = plt.subplots(figsize=(10,8)); im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(y_true.shape[1])); ax.set_yticks(range(y_true.shape[1]))
    ax.set_xticklabels(labels[:y_true.shape[1]]); ax.set_yticklabels(labels[:y_true.shape[1]])
    ax.set_title('Aggregate Confusion Matrix (argmax true vs argmax pred)')
    for i in range(y_true.shape[1]):
        for j in range(y_true.shape[1]):
            ax.text(j,i,str(int(cm[i,j])),ha='center',va='center')
    plt.colorbar(im, ax=ax); plt.tight_layout(); plt.savefig(os.path.join(save_dir,'confusion_matrix_aggregate.png'), dpi=150); plt.close()

def plot_roc_pr(y_true, y_pred_proba, save_dir, labels=DISEASE_LABELS):
    fig, ax = plt.subplots(figsize=(8,6))
    for i in range(y_true.shape[1]):
        try:
            fpr,tpr,_ = roc_curve(y_true[:,i], y_pred_proba[:,i])
            auc_val = roc_auc_score(y_true[:,i], y_pred_proba[:,i])
            ax.plot(fpr,tpr,label=f'{labels[i]} (AUC={auc_val:.3f})')
        except Exception:
            pass
    ax.plot([0,1],[0,1],'k--'); ax.set_title('ROC Curves'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(save_dir,'roc_curves.png'), dpi=150); plt.close()

    fig, ax = plt.subplots(figsize=(8,6))
    for i in range(y_true.shape[1]):
        try:
            p,r,_ = precision_recall_curve(y_true[:,i], y_pred_proba[:,i])
            ap = average_precision_score(y_true[:,i], y_pred_proba[:,i])
            ax.plot(r,p,label=f'{labels[i]} (AP={ap:.3f})')
        except Exception:
            pass
    ax.set_title('PR Curves'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(save_dir,'pr_curves.png'), dpi=150); plt.close()

def plot_metrics_summary(metrics_dict, save_dir):
    keys = ['precision_macro','recall_macro','f1_macro','roc_auc_macro','average_precision_macro']
    names = ['Precision','Recall','F1','ROC AUC','Avg Precision']
    vals = [metrics_dict.get(k,0) for k in keys]
    fig, ax = plt.subplots(figsize=(8,4))
    bars = ax.bar(names, vals)
    ax.set_ylim(0,1.05); ax.set_title('Overall Metrics (Macro)')
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.3f}', ha='center')
    plt.tight_layout(); plt.savefig(os.path.join(save_dir,'metrics_summary.png'), dpi=150); plt.close()

    f1_list = metrics_dict.get('f1_per_class', [])
    p_list = metrics_dict.get('precision_per_class', [])
    r_list = metrics_dict.get('recall_per_class', [])
    if len(f1_list) > 0:
        x = np.arange(len(f1_list)); w=0.25
        fig, ax = plt.subplots(figsize=(12,5))
        ax.bar(x-w,p_list,w,label='Precision'); ax.bar(x,r_list,w,label='Recall'); ax.bar(x+w,f1_list,w,label='F1')
        ax.set_xticks(x); ax.set_xticklabels(DISEASE_LABELS[:len(f1_list)]); ax.set_ylim(0,1.05); ax.legend()
        ax.set_title('Per-Class Precision, Recall, F1')
        plt.tight_layout(); plt.savefig(os.path.join(save_dir,'per_class_metrics.png'), dpi=150); plt.close()

def plot_feature_importance(y_pred_proba, save_dir, y_true=None):
    mean_pred = np.mean(y_pred_proba, axis=0)
    fig, ax = plt.subplots(figsize=(10,5)); x=np.arange(len(mean_pred))
    ax.bar(x-0.2, mean_pred, 0.4, label='Mean predicted probability')
    if y_true is not None:
        mean_true = np.mean((y_true>=0.5).astype(np.float32), axis=0)
        ax.bar(x+0.2, mean_true, 0.4, label='Ground-truth prevalence', alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(DISEASE_LABELS[:len(mean_pred)]); ax.set_ylim(0,1.05); ax.legend()
    ax.set_title('Feature Importance: Mean Prediction & Prevalence by Class')
    plt.tight_layout(); plt.savefig(os.path.join(save_dir,'feature_importance_by_class.png'), dpi=150); plt.close()

def run_full_evaluation(model, X_test, y_test, history_warmup=None, history_finetune=None, output_dir='outputs', X_val=None, y_val=None, use_tta=True):
    eval_dir = os.path.join(output_dir, 'evaluation'); os.makedirs(eval_dir, exist_ok=True)
    y_test = np.array(y_test); y_test_bin = (y_test>=0.5).astype(np.int32)
    y_pred_proba = predict_with_tta(model, X_test, batch_size=16, verbose=1) if use_tta else model.predict(X_test, verbose=1)
    metrics_050 = compute_all_metrics(y_test, y_pred_proba, threshold=0.5)
    bundle = {'test_metrics_threshold_0.5':metrics_050, 'use_tta':bool(use_tta)}

    thresholds_tuned = None
    if X_val is not None and y_val is not None:
        y_val_proba = predict_with_tta(model, X_val, batch_size=16, verbose=0) if use_tta else model.predict(X_val, verbose=0)
        thresholds_tuned = tune_per_class_thresholds_f1(y_val, y_val_proba)
        bundle['per_class_thresholds_tuned_on_val'] = thresholds_tuned.tolist()
        metrics = compute_all_metrics(y_test, y_pred_proba, threshold=thresholds_tuned)
        bundle['test_metrics_threshold_tuned'] = metrics
        y_pred = binarize_predictions(y_pred_proba, thresholds_tuned)
    else:
        metrics = metrics_050
        y_pred = binarize_predictions(y_pred_proba, 0.5)

    bundle['primary_reported_test_metrics'] = metrics
    with open(os.path.join(eval_dir, 'metrics.json'), 'w') as f:
        json.dump(_json_safe(bundle), f, indent=2)

    if history_warmup is not None: plot_training_history(history_warmup, eval_dir, prefix='warmup')
    if history_finetune is not None: plot_training_history(history_finetune, eval_dir, prefix='finetune')
    plot_confusion_matrices(y_test_bin, y_pred, eval_dir)
    plot_aggregate_confusion(y_test_bin, y_pred, eval_dir)
    plot_roc_pr(y_test_bin, y_pred_proba, eval_dir)
    plot_metrics_summary(metrics, eval_dir)
    plot_feature_importance(y_pred_proba, eval_dir, y_true=y_test_bin)

    print('All evaluation plots saved to:', eval_dir)
    return metrics

print('TensorFlow:', tf.__version__)

In [ ]:
# Auto-detect CSV and image folder from mounted ODIR dataset
import pandas as pd

candidate_csvs = sorted(glob.glob(os.path.join(DATASET_ROOT, '**', '*.csv'), recursive=True))
candidate_xlsx = sorted(glob.glob(os.path.join(DATASET_ROOT, '**', '*.xlsx'), recursive=True))

candidate_img_dirs = []
for d in glob.glob(os.path.join(DATASET_ROOT, '**'), recursive=True):
    if os.path.isdir(d):
        base = os.path.basename(d).lower()
        if base in {'preprocessed_images', 'training images', 'testing images', 'images', 'ben_graham_images'}:
            candidate_img_dirs.append(d)

print('CSV candidates:', len(candidate_csvs))
for p in candidate_csvs[:10]:
    print('-', p)
print('XLSX candidates:', len(candidate_xlsx))
for p in candidate_xlsx[:10]:
    print('-', p)
print('Image-dir candidates:', len(candidate_img_dirs))
for p in candidate_img_dirs[:10]:
    print('-', p)

csv_path = None
# Preferred known names
preferred_csv_names = {'final_clean_dataset.csv', 'full_df.csv', 'odir.csv', 'odir5k.csv'}
for p in candidate_csvs:
    if os.path.basename(p).lower() in preferred_csv_names:
        csv_path = p
        break
if csv_path is None and candidate_csvs:
    csv_path = candidate_csvs[0]

# If no CSV but XLSX exists, convert first workbook to CSV
if csv_path is None and candidate_xlsx:
    xlsx_path = candidate_xlsx[0]
    print('No CSV found. Converting XLSX to CSV from:', xlsx_path)
    df_x = pd.read_excel(xlsx_path)
    csv_path = '/kaggle/working/odir_from_xlsx.csv'
    df_x.to_csv(csv_path, index=False)
    print('Converted CSV:', csv_path)

# Prefer preprocessed images when available
img_dir = None
for d in candidate_img_dirs:
    if os.path.basename(d).lower() == 'preprocessed_images':
        img_dir = d
        break
if img_dir is None and candidate_img_dirs:
    img_dir = candidate_img_dirs[0]
if img_dir is None:
    img_dir = DATASET_ROOT

if csv_path is None:
    raise FileNotFoundError('No CSV/XLSX found in DATASET_ROOT.')

print('Selected CSV:', csv_path)
print('Selected image root:', img_dir)

In [ ]:
# Load and split data
X_train, X_test, y_train, y_test = load_odir_dataset(csv_path, img_dir, sample_fraction=1.0)

strat = np.argmax(y_train, axis=1)
try:
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.15, random_state=42, stratify=strat
    )
except ValueError:
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.15, random_state=42
    )

print('Train:', X_tr.shape, y_tr.shape)
print('Val:  ', X_val.shape, y_val.shape)
print('Test: ', X_test.shape, y_test.shape)

In [ ]:
# Build improved model configuration
WEIGHTS_MODE = 'none'  # 'none' for offline Kaggle, 'imagenet' when Internet is ON, 'auto' to try then fallback

model, last_conv_layer_name, base_cnn = build_concept_aware_lnn(
    input_shape=(224, 224, 3),
    num_classes=8,
    lnn_units=(256, 128),
    use_model_augmentation=False,
    weights_mode=WEIGHTS_MODE
)

label_smoothing = 0.08
pos_w = compute_pos_weights(y_tr)
train_loss = make_weighted_focal_loss(pos_w, gamma=2.0, label_smoothing=label_smoothing)

metrics = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc', multi_label=True)
]

weight_decay = 1e-4
monitor_metric = 'val_auc'
monitor_mode = 'max'

train_ds_warmup = build_tf_dataset(X_tr, y_tr, batch_size=32, shuffle=True, augment=True, repeat=False)
val_ds_warmup = build_tf_dataset_val(X_val, y_val, batch_size=32)
train_ds_finetune = build_tf_dataset(X_tr, y_tr, batch_size=16, shuffle=True, augment=True, repeat=False)
val_ds_finetune = build_tf_dataset_val(X_val, y_val, batch_size=16)

In [ ]:
# Phase 1: Warm-up
warmup_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor=monitor_metric, patience=4, restore_best_weights=True, mode=monitor_mode),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    tf.keras.callbacks.ModelCheckpoint('models/warmup_best.weights.h5', save_best_only=True, monitor=monitor_metric, mode=monitor_mode, save_weights_only=True)
]

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=weight_decay, clipnorm=1.0),
    loss=train_loss,
    metrics=metrics
)

history_warmup = model.fit(
    train_ds_warmup,
    validation_data=val_ds_warmup,
    epochs=15,
    callbacks=warmup_callbacks,
    verbose=1
)

# Phase 2: Fine-tune (unfreeze only top 4 layers)
base_cnn.trainable = True
n_unfreeze = 4
for layer in base_cnn.layers[:-n_unfreeze]:
    layer.trainable = False

finetune_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor=monitor_metric, patience=6, restore_best_weights=True, mode=monitor_mode),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint('models/concept_lnn_optimal.weights.h5', save_best_only=True, monitor=monitor_metric, mode=monitor_mode, save_weights_only=True)
]

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=weight_decay, clipnorm=1.0),
    loss=train_loss,
    metrics=metrics
)

history_finetune = model.fit(
    train_ds_finetune,
    validation_data=val_ds_finetune,
    epochs=30,
    callbacks=finetune_callbacks,
    verbose=1
)

In [ ]:
# Evaluate and print validation/test accuracy
os.makedirs('outputs/kaggle_improved', exist_ok=True)

metrics_dict = run_full_evaluation(
    model,
    X_test,
    y_test,
    history_warmup=history_warmup,
    history_finetune=history_finetune,
    output_dir='outputs/kaggle_improved',
    X_val=X_val,
    y_val=y_val,
    use_tta=True
)

val_acc = history_finetune.history.get('val_accuracy', [None])[-1]
test_acc = metrics_dict.get('accuracy', None)

print('Validation Accuracy (last finetune epoch):', val_acc)
print('Testing Accuracy (evaluation primary metric):', test_acc)
print('Saved metrics:', 'outputs/kaggle_improved/evaluation/metrics.json')

In [ ]:
# Show all generated evaluation outputs + preview key visualizations
import os, glob, json
from IPython.display import display, Image
from pathlib import Path

eval_dir = 'outputs/kaggle_improved/evaluation'
all_files = sorted(glob.glob(os.path.join(eval_dir, '*')))
print(f'Found {len(all_files)} files in {eval_dir}')
for f in all_files:
    print('-', f)

# Preview key plots inline
preview_names = [
    'warmup_loss.png',
    'warmup_accuracy.png',
    'warmup_auc.png',
    'finetune_loss.png',
    'finetune_accuracy.png',
    'finetune_auc.png',
    'metrics_summary.png',
    'per_class_metrics.png',
    'roc_curves.png',
    'pr_curves.png',
    'confusion_matrix_aggregate.png',
    'feature_importance_by_class.png'
 ]

for name in preview_names:
    path = os.path.join(eval_dir, name)
    if os.path.isfile(path):
        print('\n===', name, '===')
        display(Image(filename=path))

# Also show per-class confusion matrices
cm_paths = sorted(glob.glob(os.path.join(eval_dir, 'confusion_matrix_*.png')))
for p in cm_paths:
    if p.endswith('confusion_matrix_aggregate.png'):
        continue
    print('\n===', os.path.basename(p), '===')
    display(Image(filename=p))

# Zip outputs for easy download from Kaggle
zip_path = '/kaggle/working/kaggle_improved_outputs.zip'
!zip -r -q {zip_path} outputs/kaggle_improved
print('\nDownload zip from:', zip_path)

# Print primary metrics JSON
metrics_path = os.path.join(eval_dir, 'metrics.json')
if os.path.isfile(metrics_path):
    with open(metrics_path, 'r') as f:
        m = json.load(f)
    print('\nPrimary reported test metrics:')
    print(json.dumps(m.get('primary_reported_test_metrics', {}), indent=2))